# Capítulo 7 - Pontos Característicos de Imagem e uma Introdução à Odometria Visual

**Atividade fora do notebook. Não rode estes comandos aqui.**

O capítulo usa descritores SURF. A apostila observa que a distribuição comum do OpenCV no Raspberry Pi pode não incluir SURF. Use a versão do OpenCV instalada no capítulo 6, com suporte a `cv2.xfeatures2d.SURF_create`.

Antes de executar este notebook, rode o notebook do capítulo 6 e salve `camera_calibration_results.npz` em `pmr3502-camera-calibration/`.

## Preparação do notebook

In [3]:
from pathlib import Path
import math

import cv2
import numpy as np
from matplotlib import pyplot as plt

BASE_DIR = Path.cwd()
CALIB_PATH = BASE_DIR.parent / 'pmr3502-camera-calibration' / 'camera_calibration_results.npz'
DATA_DIR = BASE_DIR / 'dados'
DATA_DIR.mkdir(exist_ok=True)

# Valores salvos pelo notebook do capitulo 6:
# np.savez(
#     RESULTS_PATH,
#     image_size=np.array(IMAGE_SIZE),
#     mtx=mtx,
#     dist=dist,
#     rot=rot,
#     trans=trans,
#     matriz_rot=matriz_rot,
#     s=s,
#     erro_rms=ret,
#     erro_medio_reprojecao=erro_medio,
# )
calib = np.load(CALIB_PATH)
image_size = tuple(calib['image_size'].astype(int))
mtx = calib['mtx']
dist = calib['dist']
rot = calib['rot']
trans = calib['trans']
matriz_rot = calib['matriz_rot']
s = calib['s']
erro_rms = float(calib['erro_rms'])
erro_medio_reprojecao = float(calib['erro_medio_reprojecao'])

print('Parametros carregados de:', CALIB_PATH)
print('image_size:', image_size)
print('mtx:')
print(mtx)
print('dist:')
print(dist)
print('rot:')
print(rot)
print('trans:')
print(trans)
print('matriz_rot:')
print(matriz_rot)
print('s:')
print(s)
print('erro_rms:', erro_rms)
print('erro_medio_reprojecao:', erro_medio_reprojecao)

try:
    surf_teste = cv2.xfeatures2d.SURF_create(1000)
    print('SURF disponivel.')
except Exception as exc:
    print('SURF nao disponivel neste OpenCV:')
    print(exc)


Parametros carregados de: c:\Users\tomto\OneDrive\Área de Trabalho\POLI\2026\SEM 1\PMR3502 - Elementos de robotica\robotica_movel\git\Mobile_Robotics\pmr3502-camera-calibration\camera_calibration_results.npz
image_size: (np.int64(1296), np.int64(972))
mtx:
[[768.77526677   0.         659.37744219]
 [  0.         768.44654195 445.45470799]
 [  0.           0.           1.        ]]
dist:
[[ 1.86430810e+01 -1.20531976e+01  7.96864357e-04  3.49353837e-03
  -2.92886646e+00  1.89957455e+01 -5.73433355e+00 -8.25717383e+00
  -1.11933229e-02  3.88283129e-03  1.85906170e-03 -7.93216931e-04]]
rot:
[[-0.63383973]
 [-0.67195835]
 [-1.46341918]]
trans:
[[-1.59781645]
 [43.43530503]
 [39.33563789]]
matriz_rot:
[[-0.00360701  0.99969708 -0.02434627]
 [-0.67001734  0.0156573   0.74218031]
 [ 0.74233669  0.01898947  0.6697579 ]]
s:
[[-3.11938984e-01 -5.87586411e+01  6.48699555e+01]
 [ 8.74930777e+01  1.55912111e+00  1.83235819e+00]
 [-3.63508479e-02  1.10813223e+00  1.00000000e+00]]
erro_rms: 0.5241146

## 7.2 Detecção de Cantos

**Atividade fora do notebook.**

Imprima você mesmo a imagem do triângulo da Figura 7.3 em formato A4.

Depois, posicione a folha diante do robô como no capítulo 6 e capture uma imagem com a câmera. Salve a captura em:

```text
Odometria-visual/dados/triangulo_frame.jpg
```

A apostila diz que as coordenadas reais dos vértices do triângulo são $(110,148)$, $(235,148)$ e $(110,88)$, em milímetros.

### Tarefa: Imprima você mesmo a imagem do triângulo da Figura 7.3. Reproduza os resultados das Figuras 7.4, 7.5, 7.6 e 7.7. Ajuste os parâmetros necessários. Compare as coordenadas obtidas na chamada a `goodFeaturesToTrack` com os valores reais $(110,148)$, $(235,148)$, $(110,88)$. As coordenadas obtidas estarão em pixels, enquanto que os valores reais estão em mm. Lembre-se que a projeção pode modificar a escala de pixels/mm.

In [ ]:
frame_triangulo = cv2.imread(str(DATA_DIR / 'triangulo_frame.jpg'))
if frame_triangulo is None:
    raise FileNotFoundError(DATA_DIR / 'triangulo_frame.jpg')

# Reprojecao A4 com 4 pixels/mm, como na secao 6.9.1.
mtx2_a4_4x = np.float32([[4, 0, 50], [0, 4, 4 * 105], [0, 0, 1]])
dim_a4_4x = (4 * 297, 4 * 210)

map_x, map_y = cv2.initUndistortRectifyMap(mtx, dist, s, mtx2_a4_4x, dim_a4_4x, cv2.CV_32FC1)
triangulo = cv2.remap(frame_triangulo, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(triangulo, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.4 - triangulo capturado e reprojetado')
plt.axis('off')
plt.show()

In [ ]:
cinza = cv2.cvtColor(triangulo, cv2.COLOR_BGR2GRAY)
cinza_float = np.float32(cinza)

r = cv2.cornerHarris(cinza_float, 7, 3, 0.015)

arestas = r < -400
cantos_harris = r > 400

plt.figure(figsize=(12, 6))
plt.imshow(arestas, cmap='gray')
plt.title('Figura 7.5 - regioes com R < -400')
plt.axis('off')
plt.show()

plt.figure(figsize=(12, 6))
plt.imshow(cantos_harris, cmap='gray')
plt.title('Figura 7.6 - regioes com R > 400')
plt.axis('off')
plt.show()

In [ ]:
cantos = cv2.goodFeaturesToTrack(cinza, 3, 0.01, 10)
cantos = cantos.reshape((len(cantos), 2))

triangulo_cantos = np.array(triangulo)
for x, y in cantos:
    cv2.circle(triangulo_cantos, (int(x), int(y)), 8, (0, 0, 255), 2)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(triangulo_cantos, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.7 - cantos encontrados por goodFeaturesToTrack')
plt.axis('off')
plt.show()

print('Cantos em pixels:')
print(cantos)

print('Valores reais em mm citados pela apostila:')
print(np.float32([[110, 148], [235, 148], [110, 88]]))

print('Valores reais multiplicados por 4 pixels/mm, para comparar com a reprojecao A4 4x:')
print(4 * np.float32([[110, 148], [235, 148], [110, 88]]))

## 7.3 Associação de Pontos Característicos

**Atividade fora do notebook.**

Capture uma segunda imagem do triângulo depois de experimentar uma movimentação pequena do robô, por exemplo um deslocamento de aproximadamente 10 mm no eixo $y'$.

Salve a captura em:

```text
Odometria-visual/dados/triangulo_frame_2.jpg
```

### Tarefa: Reproduza os resultados da Figura 7.8. Experimente diferentes movimentações do robô. Verifique o conteúdo do vetor de associações resultante, em particular os atributos `distance`, `trainIdx` e `queryIdx`.

In [ ]:
frame_triangulo_2 = cv2.imread(str(DATA_DIR / 'triangulo_frame_2.jpg'))
if frame_triangulo_2 is None:
    raise FileNotFoundError(DATA_DIR / 'triangulo_frame_2.jpg')

triangulo_2 = cv2.remap(frame_triangulo_2, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
cinza_2 = cv2.cvtColor(triangulo_2, cv2.COLOR_BGR2GRAY)

pontos1 = cv2.goodFeaturesToTrack(cinza, 3, 0.01, 10).reshape((-1, 2))
pontos2 = cv2.goodFeaturesToTrack(cinza_2, 3, 0.01, 10).reshape((-1, 2))

matcher = cv2.BFMatcher(normType=cv2.NORM_L2SQR, crossCheck=False)
matches = matcher.match(np.float32(pontos2), np.float32(pontos1))

kp1 = np.array([cv2.KeyPoint(float(x), float(y), 3) for x, y in pontos1])
kp2 = np.array([cv2.KeyPoint(float(x), float(y), 3) for x, y in pontos2])

imagem_matches = cv2.drawMatches(triangulo, kp1, triangulo_2, kp2, matches, None, matchColor=(0, 255, 0))

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(imagem_matches, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.8 - associacao de pontos caracteristicos')
plt.axis('off')
plt.show()

for i, m in enumerate(matches):
    print(i, 'distance=', m.distance, 'trainIdx=', m.trainIdx, 'queryIdx=', m.queryIdx)

## 7.4 Pontos Característicos com Descritores

**Atividade fora do notebook.**

Recupere a imagem da borboleta e imprima em formato A4.

Comando da apostila, com formatação corrigida:

```bash
wget https://answers.opencv.org/upfiles/14442328141036795.jpg -O dados/borboleta.jpg
```

Depois, capture a imagem impressa com o robô na mesma orientação usada na seção 7.2 e salve em:

```text
Odometria-visual/dados/borboleta_frame.jpg
```

Para a Figura 7.15, gire a folha de 180 graus e salve uma nova captura em:

```text
Odometria-visual/dados/borboleta_frame_180.jpg
```

### Tarefa: Recupere a Figura da Borboleta em https://answers.opencv.org/upfiles/14442328141036795.jpg. Imprima a figura em formato A4. Reproduza os resultados das Figuras 7.10, 7.11, 7.12, 7.13, 7.14 e 7.15. Ajuste os parâmetros necessários.

Borboleta original em .jpg direto do computador

In [5]:

fly_path = DATA_DIR / 'borboleta.jpg'
fly = cv2.imdecode(np.fromfile(str(fly_path), dtype=np.uint8), cv2.IMREAD_COLOR)

if fly is None:
    raise FileNotFoundError(DATA_DIR / 'borboleta.jpg')

fly_gray = cv2.cvtColor(fly, cv2.COLOR_BGR2GRAY)

surf = cv2.xfeatures2d.SURF_create(60000)
kpfly, desc_fly = surf.detectAndCompute(fly_gray, None)

fly_keypoints = cv2.drawKeypoints(
    fly,
    kpfly,
    None,
    (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)

print('Pontos encontrados na borboleta original:', len(kpfly))
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(fly_keypoints, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.10 - pontos SURF na borboleta original')
plt.axis('off')
plt.show()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv_contrib\modules\xfeatures2d\src\surf.cpp:1028: error: (-213:The function/feature is not implemented) This algorithm is patented and is excluded in this configuration; Set OPENCV_ENABLE_NONFREE CMake option and rebuild the library in function 'cv::xfeatures2d::SURF::create'


Borboleta impressa e remapeada pela camera do robo com 1 mm por pixel e deslocamento para enquadrar a folha A4.

In [ ]:
fly_frame = cv2.imread(str(DATA_DIR / 'borboleta_frame.jpg'))
if fly_frame is None:
    raise FileNotFoundError(DATA_DIR / 'borboleta_frame.jpg')

# Reprojecao A4 especifica da borboleta: 1 pixel/mm, com origem y no centro da folha.
# O deslocamento x=0 enquadra a folha mais para a esquerda que o mapa de 4 px/mm usado antes.
pixels_por_mm_fly = 1
deslocamento_x_fly_mm = 0
deslocamento_y_fly_mm = 105
mtx2_a4_fly = np.float32([
    [pixels_por_mm_fly, 0, pixels_por_mm_fly * deslocamento_x_fly_mm],
    [0, pixels_por_mm_fly, pixels_por_mm_fly * deslocamento_y_fly_mm],
    [0, 0, 1],
])
dim_a4_fly = (297 * pixels_por_mm_fly, 210 * pixels_por_mm_fly)
map_x_fly, map_y_fly = cv2.initUndistortRectifyMap(mtx, dist, s, mtx2_a4_fly, dim_a4_fly, cv2.CV_32FC1)

flycapture = cv2.remap(fly_frame, map_x_fly, map_y_fly, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
flycapture_gray = cv2.cvtColor(flycapture, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(flycapture, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.11 - borboleta capturada e remapeada')
plt.axis('off')
plt.show()

surf.setHessianThreshold(10000)
kpcapture, desc_capture = surf.detectAndCompute(flycapture_gray, None)
flycapture_keypoints = cv2.drawKeypoints(
    flycapture,
    kpcapture,
    None,
    (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)

print('Pontos encontrados na captura:', len(kpcapture))
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(flycapture_keypoints, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.12 - pontos SURF na borboleta capturada')
plt.axis('off')
plt.show()

Associacao direta por forca bruta entre as duas imagens:

In [ ]:
matcher = cv2.BFMatcher(cv2.NORM_L2)
matches_forca_bruta = matcher.match(desc_fly, desc_capture)
matches_forca_bruta = sorted(matches_forca_bruta, key=lambda match: match.distance)

imagem_forca_bruta = cv2.drawMatches(
    fly,
    kpfly,
    flycapture,
    kpcapture,
    matches_forca_bruta,
    None,
    matchColor=(0, 255, 0),
    singlePointColor=(255, 0, 0),
    flags=cv2.DrawMatchesFlags_DEFAULT,
)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(imagem_forca_bruta, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.13 - associacao direta por forca bruta')
plt.axis('off')
plt.show()

Matches entre as duas imagens com teste de Lowe:

In [ ]:
matches = matcher.knnMatch(desc_fly, desc_capture, k=2)

matchesMask = [[0, 0] for i in range(len(matches))]
for i, pair in enumerate(matches):
    if len(pair) == 2:
        m, n = pair
        if m.distance < 0.7 * n.distance:
            matchesMask[i] = [1, 0]

draw_params = dict(
    matchColor=(0, 255, 0),
    singlePointColor=(255, 0, 0),
    matchesMask=matchesMask,
    flags=cv2.DrawMatchesFlags_DEFAULT,
)
imagem_lowe = cv2.drawMatchesKnn(fly, kpfly, flycapture, kpcapture, matches, None, **draw_params)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(imagem_lowe, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.14 - associacoes filtradas pelo teste de Lowe')
plt.axis('off')
plt.show()

Match com a botboleta rotacionada

In [ ]:
fly_frame_180 = cv2.imread(str(DATA_DIR / 'borboleta_frame_180.jpg'))
if fly_frame_180 is None:
    raise FileNotFoundError(DATA_DIR / 'borboleta_frame_180.jpg')

flycapture_180 = cv2.remap(fly_frame_180, map_x_fly, map_y_fly, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
flycapture_180_gray = cv2.cvtColor(flycapture_180, cv2.COLOR_BGR2GRAY)

surf.setHessianThreshold(10000)
kpcapture_180, desc_capture_180 = surf.detectAndCompute(flycapture_180_gray, None)
matches_180 = matcher.knnMatch(desc_fly, desc_capture_180, k=2)

matchesMask_180 = [[0, 0] for i in range(len(matches_180))]
for i, pair in enumerate(matches_180):
    if len(pair) == 2:
        m, n = pair
        if m.distance < 0.7 * n.distance:
            matchesMask_180[i] = [1, 0]

draw_params_180 = dict(
    matchColor=(0, 255, 0),
    singlePointColor=(255, 0, 0),
    matchesMask=matchesMask_180,
    flags=cv2.DrawMatchesFlags_DEFAULT,
)
imagem_180 = cv2.drawMatchesKnn(fly, kpfly, flycapture_180, kpcapture_180, matches_180, None, **draw_params_180)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(imagem_180, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.15 - associacoes entre imagens com orientacoes distintas')
plt.axis('off')
plt.show()

## 7.5 Estimando transformações bidimensionais entre imagens

**Atividade fora do notebook.**

Recupere as figuras do Campus da USP e da Poli:

```bash
wget https://www.lsc.poli.usp.br/pmr3502/ciduniv.png -O dados/ciduniv.png
wget https://www.lsc.poli.usp.br/pmr3502/poli.png -O dados/poli.png
```

Imprima a figura do campus da Poli em formato A4. Capture a impressão com o robô e salve em:

```text
Odometria-visual/dados/poli_frame.jpg
```

### Tarefa: Recupere as Figuras do Campus da USP e da Poli em https://www.lsc.poli.usp.br/pmr3502/ciduniv.png e https://www.lsc.poli.usp.br/pmr3502/poli.png. Imprima a Figura do campus da Poli em formato A4.

1. Reproduza os resultados das Figuras 7.17, 7.20, 7.21 e 7.22. Ajuste os parâmetros necessários.
2. Considerando a translação e escala do remapeamento da imagem do campus da Poli, quais as coordenadas nesta imagem da origem do sistema global de coordenadas?
3. Dado o resultado do RANSAC, quais as coordenadas na imagem do campus da USP correspondentes à origem do sistema global de coordenadas?

In [ ]:
campus = cv2.imread(str(DATA_DIR / 'ciduniv.png'))
poli_ref = cv2.imread(str(DATA_DIR / 'poli.png'))
poli_frame = cv2.imread(str(DATA_DIR / 'poli_frame.jpg'))

if campus is None:
    raise FileNotFoundError(DATA_DIR / 'ciduniv.png')
if poli_ref is None:
    raise FileNotFoundError(DATA_DIR / 'poli.png')
if poli_frame is None:
    raise FileNotFoundError(DATA_DIR / 'poli_frame.jpg')

surf.setHessianThreshold(10000)
kp_campus, des_campus = surf.detectAndCompute(cv2.cvtColor(campus, cv2.COLOR_BGR2GRAY), None)
campus_kp = cv2.drawKeypoints(campus, kp_campus, None, (255, 0, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

print('Pontos no campus:', len(kp_campus))
plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(campus_kp, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.17 - pontos SURF na imagem do campus')
plt.axis('off')
plt.show()

In [ ]:
# Remapeamento da captura da Poli com resolucao 4 pixels/mm.
mtx2_poli = np.float32([[4, 0, 0], [0, 4, -4 * 105], [0, 0, 1]])
dim_poli = (4 * 297, 4 * 210)
map_x_poli, map_y_poli = cv2.initUndistortRectifyMap(mtx, dist, s, mtx2_poli, dim_poli, cv2.CV_32FC1)
poli_capture = cv2.remap(poli_frame, map_x_poli, map_y_poli, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

surf.setHessianThreshold(4000)
kp_poli, des_poli = surf.detectAndCompute(cv2.cvtColor(poli_capture, cv2.COLOR_BGR2GRAY), None)
poli_kp = cv2.drawKeypoints(poli_capture, kp_poli, None, (255, 0, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

print('Pontos na captura da Poli:', len(kp_poli))
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(poli_kp, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.20 - pontos SURF na impressao do campus da Poli')
plt.axis('off')
plt.show()

In [ ]:
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=4)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
flann.add(np.array([des_campus]))
flann.train()

matches_poli = flann.knnMatch(des_poli, k=2)

boas = []
for pair in matches_poli:
    if len(pair) == 2:
        m, n = pair
        if m.distance < 0.7 * n.distance:
            boas.append(m)

print('Associacoes apos teste de Lowe:', len(boas))
imagem_flann = cv2.drawMatches(campus, kp_campus, poli_capture, kp_poli, boas, None, matchColor=(0, 255, 0))

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(imagem_flann, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.21 - associacoes encontradas com os pontos da imagem do Campus')
plt.axis('off')
plt.show()

In [ ]:
pts_campus = np.float32([kp_campus[m.trainIdx].pt for m in boas])
pts_poli = np.float32([kp_poli[m.queryIdx].pt for m in boas])

matriz_afim, mascara_ransac = cv2.estimateAffinePartial2D(pts_poli, pts_campus)
print('Matriz estimateAffinePartial2D:')
print(matriz_afim)
print('Associacoes mantidas pelo RANSAC:', int(mascara_ransac.sum()))

matches_ransac = [m for m, keep in zip(boas, mascara_ransac.ravel()) if keep]
imagem_ransac = cv2.drawMatches(campus, kp_campus, poli_capture, kp_poli, matches_ransac, None, matchColor=(0, 255, 0))

# Desenha a posicao estimada dos cantos da imagem da Poli no campus completo.
h_poli, w_poli = poli_capture.shape[:2]
cantos_poli = np.float32([[0, 0], [w_poli, 0], [w_poli, h_poli], [0, h_poli]]).reshape(-1, 1, 2)
cantos_campus = cv2.transform(cantos_poli, matriz_afim).astype(int)
for i in range(4):
    p1 = tuple(cantos_campus[i, 0])
    p2 = tuple(cantos_campus[(i + 1) % 4, 0])
    cv2.line(imagem_ransac, p1, p2, (0, 0, 255), 5)

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(imagem_ransac, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.22 - associacoes que correspondem a rotacao, translacao e escala')
plt.axis('off')
plt.show()

origem_global_h = mtx2_poli @ np.float32([0, 0, 1])
origem_global_na_poli = np.float32([origem_global_h[:2] / origem_global_h[2]])
origem_global_no_campus = cv2.transform(origem_global_na_poli.reshape(-1, 1, 2), matriz_afim).reshape(-1, 2)

print('Coordenadas da origem do sistema global na imagem da Poli:')
print(origem_global_na_poli)
print('Coordenadas correspondentes na imagem do campus da USP:')
print(origem_global_no_campus)

## 7.6 Explorando a Área Visível

**Atividade fora do notebook.**

Para calcular a máscara, fotografe em uma área bem iluminada uma folha de papel branco colocada à frente do robô. Salve a imagem em:

```text
Odometria-visual/dados/folha_branca_frame.jpg
```

### Tarefa:

1. Calcule a matriz $Q$ da reprojeção completa e o mapa correspondente via `initUndistortRectifyMap`. Reproduza o resultado da Figura 7.32a.
2. Calcule a máscara de pontos visíveis no solo. Reproduza o resultado da Figura 7.37.

In [ ]:
# Reprojecao de uma regiao maior no solo: x de 0 a 600 mm, y de -0.8*xmax a +0.8*xmax.
xmax = 600
ymax = 0.8 * xmax
pixels_por_mm = 2
cx = 0
cy = -ymax
w = xmax
h = 2 * ymax

Q = np.float32([
    [pixels_por_mm, 0, -pixels_por_mm * cx],
    [0, pixels_por_mm, -pixels_por_mm * cy],
    [0, 0, 1],
])
dim_visivel = (int(pixels_por_mm * w), int(pixels_por_mm * h))

map_x_vis, map_y_vis = cv2.initUndistortRectifyMap(mtx, dist, s, Q, dim_visivel, cv2.CV_32FC1)
imagem_reprojetada = cv2.remap(poli_frame, map_x_vis, map_y_vis, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

surf.setHessianThreshold(4000)
kp_sem_mascara, desc_sem_mascara = surf.detectAndCompute(cv2.cvtColor(imagem_reprojetada, cv2.COLOR_BGR2GRAY), None)
figura_732a = cv2.drawKeypoints(imagem_reprojetada, kp_sem_mascara, None, (255, 0, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(figura_732a, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.32a - pontos caracteristicos com artefatos de reprojecao')
plt.axis('off')
plt.show()

print('Matriz Q:')
print(Q)

In [ ]:
folha = cv2.imread(str(DATA_DIR / 'folha_branca_frame.jpg'))
if folha is None:
    raise FileNotFoundError(DATA_DIR / 'folha_branca_frame.jpg')

frame = cv2.cvtColor(folha, cv2.COLOR_BGR2GRAY)
frame[:-frame.shape[0] // 4, :] = 255

ret, frame_limiarizado = cv2.threshold(frame, 190, 255, cv2.THRESH_BINARY)
kernel = np.ones((5, 5), np.uint8)
closing = cv2.morphologyEx(frame_limiarizado, cv2.MORPH_CLOSE, kernel)
erosion = cv2.erode(closing, kernel, iterations=2)

mask = cv2.remap(erosion, map_x_vis, map_y_vis, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
mask[-250:, :100] = False

kp_mascara, desc_mascara = surf.detectAndCompute(cv2.cvtColor(imagem_reprojetada, cv2.COLOR_BGR2GRAY), mask)
figura_737 = cv2.drawKeypoints(imagem_reprojetada, kp_mascara, None, (255, 0, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

plt.figure(figsize=(12, 8))
plt.imshow(mask, cmap='gray')
plt.title('Mascara de pontos visiveis no solo')
plt.axis('off')
plt.show()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(figura_737, cv2.COLOR_BGR2RGB))
plt.title('Figura 7.37 - descritores em cena mascarada')
plt.axis('off')
plt.show()

## 7.7 Uma Aplicação em Odometria Visual

### Tarefa:

1. Se a movimentação relativa no instante $i$, no referencial do robô, foi de $\delta x'_i$, $\delta y'_i$, $\delta \theta'_i$, e se $x_i$, $y_i$, $\theta_i$ é a posição e orientação do robô em um referencial externo, então:

$$
x_i = x_{i-1} + \delta x'_i \cos\theta_{i-1} - \delta y'_i \sin\theta_{i-1}
$$

$$
y_i = y_{i-1} + \delta y'_i \cos\theta_{i-1} + \delta x'_i \sin\theta_{i-1}
$$

$$
\theta_i = \theta_{i-1} + \delta	heta'_i
$$

O código em https://gitlab.uspdigital.usp.br/thiago/pmr3502-odometria-visual.git implementa parcialmente um serviço de estimativa de posição do robô baseado em movimentação relativa entre quadros. Você deve completar este código implementando a classe `ProcessamentoVisao` e seus métodos `__init__`, `primeiroQuadro` e `estimaMovimento`.

O método `__init__(self, largura, altura)` inicializa o objeto. O parâmetro `largura` é a largura, em pixels, da imagem capturada pela câmera; `altura` é a altura, em pixels, da imagem capturada pela câmera.

O método `primeiroQuadro(self, quadro)` é invocado quando o primeiro quadro é capturado pela câmera.

O método `estimaMovimento(self, quadro)` é invocado quando a câmera do robô captura um novo quadro. Ele deve retornar uma tupla com 3 componentes, respectivamente a movimentação em $x'$, $y'$ e $\theta$ em relação ao quadro anterior.

Complete o programa e execute-o com o comando `servidor.py`. O programa oferece uma interface gráfica via web browser mostrando a estimativa de posição que acumula sucessivos movimentos relativos do robô.

Você pode movimentar o robô usando, por exemplo, o programa de telepresença construído na seção 2.5. Evite manobrar o robô na velocidade máxima: é possível que não haja associações suficientes entre quadros sucessivos para estimar movimento relativo. Observe como erros de estimativa acumulam-se ao longo do tempo.

Sugestões:

- Esta tarefa requer uma superfície com textura característica para que o robô possa reconstruir o movimento. Pisos com cores homogêneas não são recomendados.
- Você deve construir pontos característicos SURF a cada quadro e associá-los aos pontos do quadro anterior. Isso significa que, ao final do processamento de uma imagem, você deve guardar os pontos característicos para o próximo quadro.
- Maximize a área de visibilidade usando a matriz de câmera construída aqui. Uma escala de 1 pixel/mm deve ser suficiente.
- Use um valor de `threshold` que produza aproximadamente 100 pontos característicos por quadro.
- Use o filtro de Lowe para eliminar associações ambíguas. Você precisa de um mínimo de 4 boas associações para estimar o movimento, mas o ideal é ter ao menos 8. Mostre um alerta caso em alguma iteração não consiga obter 8 boas associações.
- A estimativa de movimento pode ser estimada por `estimateAffinePartial2D`, mas você deve modificar a matriz resultante de acordo com a equação (7.6). Verifique se a máscara resultante retornada por `estimateAffinePartial2D` usa ao menos 5 associações. Note, no entanto, que a estimativa de movimento dada por `estimateAffinePartial2D` é a do mundo em relação ao robô. O que você quer realmente é o movimento do robô em relação ao mundo. Mostre um alerta se esse valor não for obtido.
- Caso não consiga estimar o movimento, por exemplo, por falta de associações suficientes, retorne $(0,0,0)$. De todo modo, lembre-se de armazenar os descritores para o próximo quadro.
- A criação de descritores SURF pode fazer o regulador de clock do seu Raspberry Pi exceder a potência disponível pelas suas baterias. Isso pode fazer com que seu Raspberry Pi reinicie espontaneamente. Se isso acontecer, adicione a linha `arm_freq=1000` ao começo do arquivo `/boot/firmware/config.txt` e reinicie o Raspberry Pi. Isso irá limitar o clock máximo do seu Raspberry Pi a 1 GHz. Se o problema persistir, você pode reduzir ainda mais esse limite, mas recomenda-se manter ao menos 700 MHz para essa atividade.

**Atividade fora do notebook.**

Recupere o código do serviço:

```bash
git clone https://gitlab.uspdigital.usp.br/thiago/pmr3502-odometria-visual.git
```

Complete a classe `ProcessamentoVisao` no projeto clonado. Execute o serviço com:

```bash
python3 servidor.py
```

Se a Raspberry Pi reiniciar espontaneamente durante a criação de descritores SURF, a apostila recomenda adicionar a linha abaixo ao começo de `/boot/firmware/config.txt` e reiniciar:

```text
arm_freq=1000
```

Célula de referência para a equação de acumulação de pose citada na tarefa.

In [ ]:
x_anterior = 0.0
y_anterior = 0.0
theta_anterior = 0.0

delta_x_linha = 0.0
delta_y_linha = 0.0
delta_theta_linha = 0.0

x = x_anterior + delta_x_linha * math.cos(theta_anterior) - delta_y_linha * math.sin(theta_anterior)
y = y_anterior + delta_y_linha * math.cos(theta_anterior) + delta_x_linha * math.sin(theta_anterior)
theta = theta_anterior + delta_theta_linha

print(x, y, theta)